# BrailleLens — Fingertip YOLO26n (Kaggle **GPU**, not TPU)

Train a **single-class** `fingertip` detector for tip-only views (phone / future smart glasses).

## Accelerator choice

| Option | Use? | Why |
|--------|------|-----|
| **Kaggle GPU** (P100 / T4) | **Yes — recommended** | Ultralytics YOLO26 is **PyTorch + CUDA** |
| Local PC (CPU-only) | Smoke tests only | Full train will be very slow |
| **Kaggle TPU v5e-8** | **No** | YOLO/Ultralytics does **not** train natively on TPU |

In Kaggle: **Settings → Accelerator → GPU** (not TPU).

## Dataset prep (on your PC first)

```powershell
finger_cell_track\.venv\Scripts\python.exe finger_cell_track/prepare_tip_yolo_dataset.py --clean
# then zip finger_cell_track/datasets/fingertip_yolo26 → upload to Kaggle as a Dataset
```

In [ ]:
# Verify GPU (must be True for real training)
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No CUDA GPU. Switch Kaggle accelerator to GPU, or expect a very slow CPU run.')

In [ ]:
%pip install -q ultralytics
from ultralytics import YOLO
import ultralytics
print('ultralytics', ultralytics.__version__)

In [ ]:
from pathlib import Path
import os

# --- edit these paths ---
# After adding your Kaggle Dataset, it appears under /kaggle/input/<dataset-name>/
CANDIDATES = [
    Path('/kaggle/input/fingertip-yolo26/fingertip_yolo26'),
    Path('/kaggle/input/fingertip-yolo26'),
    Path('/kaggle/working/fingertip_yolo26'),
]

DATA_DIR = None
for p in CANDIDATES:
    if (p / 'data.yaml').exists():
        DATA_DIR = p
        break
    # yaml might be nested one level
    if p.exists():
        hits = list(p.rglob('data.yaml'))
        if hits:
            DATA_DIR = hits[0].parent
            break

assert DATA_DIR is not None, (
    'data.yaml not found. Upload zipped fingertip_yolo26 as a Kaggle Dataset '
    'and add it to this notebook, then fix CANDIDATES.'
)
DATA_YAML = DATA_DIR / 'data.yaml'
print('Using', DATA_YAML)
print(DATA_YAML.read_text()[:400])

In [ ]:
# Fix path: inside Kaggle, rewrite data.yaml to absolute path of DATA_DIR
text = DATA_YAML.read_text()
lines = []
for line in text.splitlines():
    if line.startswith('path:'):
        lines.append(f'path: {DATA_DIR.resolve().as_posix()}')
    else:
        lines.append(line)
DATA_YAML.write_text('\n'.join(lines) + '\n')
print(DATA_YAML.read_text())

In [ ]:
# Hyperparameters — tip is a small object; 640 imgsz is a good start
EPOCHS = 50          # raise to 80–100 if val mAP still climbing
IMGSZ = 640
BATCH = 16           # lower to 8 if OOM on T4
MODEL = 'yolo26n.pt' # nano: fast enough for phone/glasses later
PROJECT = '/kaggle/working/runs/fingertip'
NAME = 'yolo26n_tip'

model = YOLO(MODEL)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0 if torch.cuda.is_available() else 'cpu',
    project=PROJECT,
    name=NAME,
    exist_ok=True,
    patience=15,
    plots=True,
)
print('done', results)

In [ ]:
from IPython.display import Image, display

best = Path(PROJECT) / NAME / 'weights' / 'best.pt'
print('best weights:', best, 'exists=', best.exists())

val = YOLO(str(best))
metrics = val.val(data=str(DATA_YAML), imgsz=IMGSZ)
print(metrics)

for plot_name in ('results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg'):
    p = Path(PROJECT) / NAME / plot_name
    if p.exists():
        display(Image(filename=str(p)))

In [ ]:
# Copy best.pt to an easy download path
import shutil
out = Path('/kaggle/working/yolo26n_fingertip_best.pt')
shutil.copy2(best, out)
print('Download from:', out)
print('Then place on PC as: finger_cell_track/weights/yolo26n_fingertip_best.pt')

## After training

1. Download `yolo26n_fingertip_best.pt` from Kaggle output.
2. Save to `finger_cell_track/weights/yolo26n_fingertip_best.pt`.
3. Next step (code change): swap MediaPipe in `live_app.py` / `hand_track.py` for this YOLO tip model.
4. Later: fine-tune again on your own tip-on-Braille IP Webcam frames.